# **Eksperimen SML - Ahmad Asyhadi**

Notebook ini dibuat menggunakan **Template Eksperimen MSML** dan memuat seluruh tahapan eksperimen secara manual, mulai dari perkenalan dataset, import library, data loading, Exploratory Data Analysis (EDA), hingga preprocessing dataset.


# **1. Perkenalan Dataset**


Dataset yang digunakan adalah **Breast Cancer Wisconsin Dataset**. Dataset ini digunakan untuk klasifikasi diagnosis kanker payudara berdasarkan karakteristik sel hasil pemeriksaan.

Tujuan eksperimen ini adalah membangun dataset hasil preprocessing yang siap digunakan untuk pelatihan model machine learning.

**Informasi dataset:**

- Nama dataset: Breast Cancer Wisconsin Dataset
- Jenis masalah: Classification
- Target: `diagnosis`
- Jumlah fitur utama: 30 fitur numerik
- Label target:
  - `M` = Malignant
  - `B` = Benign

Pada tahap preprocessing, label target akan dikonversi menjadi bentuk numerik:

- `M` menjadi `0`
- `B` menjadi `1`

Dataset mentah disimpan dalam folder:

```text
../namadataset_raw/breast_cancer_raw.csv
```

Dataset hasil preprocessing akan disimpan dalam folder:

```text
namadataset_preprocessing/breast_cancer_preprocessing.csv
```


# **2. Import Library**


Pada tahap ini, beberapa library Python yang diperlukan akan diimpor. Library digunakan untuk membaca data, melakukan eksplorasi data, visualisasi, preprocessing, dan pembagian dataset.


In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


# **3. Memuat Dataset**


Pada tahap ini dataset mentah dimuat menggunakan `pandas`. Path dibuat fleksibel agar notebook bisa dijalankan dari folder `preprocessing` maupun dari root project.


In [ ]:
# Path dataset mentah
possible_paths = [
    Path("../namadataset_raw/breast_cancer_raw.csv"),
    Path("Eksperimen_SML_AhmadAsyhadi/namadataset_raw/breast_cancer_raw.csv"),
    Path("breast_cancer_raw.csv")
]

DATA_PATH = None
for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "File breast_cancer_raw.csv tidak ditemukan. "
        "Pastikan dataset berada di folder namadataset_raw."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset berhasil dimuat dari: {DATA_PATH}")
df.head()


In [ ]:
# Melihat ukuran dataset
print("Jumlah baris dan kolom:", df.shape)


In [ ]:
# Melihat nama kolom dataset
df.columns.tolist()


# **4. Exploratory Data Analysis (EDA)**

Tahap EDA dilakukan untuk memahami struktur dataset, tipe data, data kosong, distribusi target, serta hubungan antar fitur.


## 4.1 Informasi Struktur Dataset


In [ ]:
df.info()


## 4.2 Statistik Deskriptif


In [ ]:
df.describe()


## 4.3 Pemeriksaan Missing Value


In [ ]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]


In [ ]:
print("Total missing value:", df.isnull().sum().sum())


## 4.4 Pemeriksaan Data Duplikat


In [ ]:
duplicate_count = df.duplicated().sum()
print("Jumlah data duplikat:", duplicate_count)


## 4.5 Distribusi Target


In [ ]:
df["diagnosis"].value_counts()


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="diagnosis")
plt.title("Distribusi Target Diagnosis")
plt.xlabel("Diagnosis")
plt.ylabel("Jumlah Data")
plt.show()


## 4.6 Korelasi Fitur Numerik


In [ ]:
# Menghapus kolom id jika tersedia agar tidak mengganggu analisis korelasi
eda_df = df.copy()
if "id" in eda_df.columns:
    eda_df = eda_df.drop(columns=["id"])

# Encoding sementara target untuk korelasi
eda_df["diagnosis"] = eda_df["diagnosis"].map({"M": 0, "B": 1})

plt.figure(figsize=(18, 14))
sns.heatmap(eda_df.corr(), cmap="coolwarm", linewidths=0.2)
plt.title("Correlation Heatmap")
plt.show()


## 4.7 Distribusi Beberapa Fitur Penting


In [ ]:
important_features = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness"
]

for feature in important_features:
    if feature in df.columns:
        plt.figure(figsize=(6, 4))
        sns.histplot(data=df, x=feature, hue="diagnosis", kde=True)
        plt.title(f"Distribusi {feature}")
        plt.show()


# **5. Data Preprocessing**

Tahap preprocessing dilakukan secara manual di notebook sesuai template eksperimen. Proses preprocessing meliputi:

1. Menghapus kolom yang tidak diperlukan.
2. Menghapus data duplikat jika ada.
3. Mengonversi target kategorikal menjadi numerik.
4. Memisahkan fitur dan target.
5. Melakukan train-test split.
6. Melakukan feature scaling.
7. Menyimpan dataset hasil preprocessing.


## 5.1 Menyalin Dataset


In [ ]:
df_clean = df.copy()
df_clean.head()


## 5.2 Menghapus Kolom Tidak Diperlukan


In [ ]:
# Kolom id tidak digunakan untuk pemodelan karena hanya berfungsi sebagai identifier
if "id" in df_clean.columns:
    df_clean = df_clean.drop(columns=["id"])

# Beberapa dataset breast cancer memiliki kolom kosong bernama Unnamed: 32
if "Unnamed: 32" in df_clean.columns:
    df_clean = df_clean.drop(columns=["Unnamed: 32"])

df_clean.head()


## 5.3 Menghapus Data Duplikat


In [ ]:
before_drop = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
after_drop = df_clean.shape[0]

print("Jumlah data sebelum drop duplicate:", before_drop)
print("Jumlah data setelah drop duplicate:", after_drop)


## 5.4 Encoding Target


In [ ]:
# Encoding target diagnosis
# M = Malignant -> 0
# B = Benign -> 1

df_clean["diagnosis"] = df_clean["diagnosis"].map({"M": 0, "B": 1})

df_clean["diagnosis"].value_counts()


## 5.5 Memisahkan Fitur dan Target


In [ ]:
X = df_clean.drop(columns=["diagnosis"])
y = df_clean["diagnosis"]

print("Jumlah fitur:", X.shape[1])
print("Jumlah data:", X.shape[0])


## 5.6 Train-Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


## 5.7 Feature Scaling


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

X_train_scaled.head()


## 5.8 Menggabungkan Dataset Hasil Preprocessing


In [ ]:
train_preprocessed = X_train_scaled.copy()
train_preprocessed["diagnosis"] = y_train.reset_index(drop=True)

test_preprocessed = X_test_scaled.copy()
test_preprocessed["diagnosis"] = y_test.reset_index(drop=True)

preprocessed_df = pd.concat(
    [train_preprocessed, test_preprocessed],
    axis=0,
    ignore_index=True
)

preprocessed_df.head()


In [ ]:
preprocessed_df.shape


## 5.9 Menyimpan Dataset Hasil Preprocessing


In [ ]:
# Membuat folder output jika belum tersedia
output_dir = Path("namadataset_preprocessing")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "breast_cancer_preprocessing.csv"
preprocessed_df.to_csv(output_path, index=False)

print(f"Dataset hasil preprocessing berhasil disimpan ke: {output_path}")


# **6. Kesimpulan**

Eksperimen dataset telah dilakukan menggunakan Template Eksperimen MSML. Seluruh tahapan utama telah dilakukan secara manual dalam notebook, yaitu:

- Memuat dataset mentah.
- Melakukan EDA.
- Membersihkan data.
- Melakukan encoding target.
- Melakukan train-test split.
- Melakukan scaling fitur numerik.
- Menyimpan dataset hasil preprocessing.

Dataset hasil preprocessing siap digunakan pada tahap pembangunan model machine learning dan workflow otomatis.
